# Custom Feature Generators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forge-features/forge/blob/main/notebooks/02_custom_generators.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/forge-features/forge/main?labpath=notebooks/02_custom_generators.ipynb)

This notebook shows how to use individual feature generators for fine-grained control.

## What you'll learn

1. Using numeric generators (interactions, polynomials, transformations)
2. Using categorical generators (encoders, combinations)
3. Using temporal generators (date parts, lags, rolling windows)
4. Chaining generators with ForgePipeline

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

## Numeric Feature Generators

### Interaction Features

Create features by combining numeric columns:

In [ ]:
from forge.generators.numeric import InteractionGenerator

# Sample data
df = pd.DataFrame({
    'price': [10, 20, 30, 40, 50],
    'quantity': [5, 3, 8, 2, 6],
    'discount': [0.1, 0.2, 0.15, 0.05, 0.1]
})

# Create interaction features
gen = InteractionGenerator(
    columns=['price', 'quantity'],
    operations=['multiply', 'divide', 'add']
)

result = gen.fit_transform(df)
print("Original columns:", list(df.columns))
print("After interactions:", list(result.columns))
result

### Polynomial Features

Generate polynomial and interaction terms:

In [ ]:
from forge.generators.numeric import PolynomialGenerator

gen = PolynomialGenerator(
    columns=['price', 'quantity'],
    degree=2,
    interaction_only=False
)

result = gen.fit_transform(df)
print("Polynomial features:")
result

### Numeric Transformations

Apply mathematical transformations:

In [ ]:
from forge.generators.numeric import LogTransformer, BinningTransformer

# Log transformation
log_gen = LogTransformer(columns=['price'])
result = log_gen.fit_transform(df)
print("Log transformed:")
print(result[['price', 'price_log']].head())

# Binning
bin_gen = BinningTransformer(columns=['price'], n_bins=3)
result = bin_gen.fit_transform(df)
print("\nBinned:")
print(result[['price', 'price_bin']].head())

## Categorical Feature Generators

### Encoding Methods

In [ ]:
from forge.generators.categorical import OneHotEncoder, TargetEncoder, FrequencyEncoder

# Sample categorical data
df_cat = pd.DataFrame({
    'color': ['red', 'blue', 'red', 'green', 'blue', 'red'],
    'size': ['S', 'M', 'L', 'M', 'S', 'L'],
    'value': [100, 200, 150, 175, 125, 180]
})
y = pd.Series([1, 0, 1, 0, 1, 1])

# One-Hot Encoding
ohe = OneHotEncoder(columns=['color'])
result = ohe.fit_transform(df_cat)
print("One-Hot Encoded:")
print(result.head())

In [ ]:
# Target Encoding (uses target variable)
te = TargetEncoder(columns=['color'])
result = te.fit_transform(df_cat, y)
print("Target Encoded:")
print(result[['color', 'color_target_enc']].head())

In [ ]:
# Frequency Encoding
fe = FrequencyEncoder(columns=['color'])
result = fe.fit_transform(df_cat)
print("Frequency Encoded:")
print(result[['color', 'color_freq']].head())

## Temporal Feature Generators

Extract features from datetime columns:

In [ ]:
from forge.generators.temporal import DateTimeComponents, LagGenerator

# Sample temporal data
df_time = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-01', periods=10, freq='D'),
    'sales': [100, 120, 90, 150, 200, 180, 160, 140, 170, 190]
})

# Extract date components
dt_gen = DateTimeComponents(columns=['timestamp'])
result = dt_gen.fit_transform(df_time)
print("Date components:")
print(result.head())

In [ ]:
# Create lag features
lag_gen = LagGenerator(columns=['sales'], lags=[1, 2, 3])
result = lag_gen.fit_transform(df_time)
print("Lag features:")
print(result[['sales', 'sales_lag_1', 'sales_lag_2', 'sales_lag_3']])

## Chaining Generators with ForgePipeline

Combine multiple generators into a single pipeline:

In [ ]:
from forge.transformers import ForgePipeline
from forge.generators.numeric import InteractionGenerator, PolynomialGenerator
from forge.generators.categorical import TargetEncoder

# Mixed data
df_mixed = pd.DataFrame({
    'price': [10, 20, 30, 40, 50],
    'quantity': [5, 3, 8, 2, 6],
    'category': ['A', 'B', 'A', 'C', 'B']
})
y = pd.Series([0, 1, 1, 0, 1])

# Create pipeline
pipeline = ForgePipeline([
    ('interactions', InteractionGenerator(
        columns=['price', 'quantity'],
        operations=['multiply']
    )),
    ('encoding', TargetEncoder(columns=['category'])),
    ('polynomials', PolynomialGenerator(
        columns=['price', 'quantity'],
        degree=2
    )),
])

result = pipeline.fit_transform(df_mixed, y)
print(f"Original features: {df_mixed.shape[1]}")
print(f"After pipeline: {result.shape[1]}")
print("\nFeature names:")
for col in result.columns:
    print(f"  - {col}")

## Next Steps

- [03_feature_selection.ipynb](03_feature_selection.ipynb) - Learn about feature selection methods